# MCP and Agent Interoperability

> **The story.** The Model Context Protocol standardizes discovery and invocation between AI applications and external capability servers. Its value is reducing bespoke integration glue while keeping schemas and lifecycle visible.
>
> **Where you are.** OrderFlow hard-codes inventory and pricing functions into every agent. Adding a second pricing provider requires agent-core edits.
>
> **Notation.** $C$ is the client; $S$ is an MCP server; $K(S)$ is its discovered capability set; each JSON-RPC request carries correlation ID $id$ and method $m$.

## 0 - The Challenge

> **The mission:** discover and use a second pricing provider through configuration alone, while rejecting an incompatible schema before any financial action.

```mermaid
flowchart LR
    A["Agent core"] --> G1["Bespoke inventory glue"]
    A --> G2["Bespoke pricing glue"]
    G1 --> M["MCP discovery + schemas"]
    G2 --> M
    M --> S["Config-selected providers"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G1 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G2 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Roadmap | Failure to expose | Measured unlock |
|---|---|---|
| Name the capability layers | Tool, skill, plugin, channel plugin, and MCP server blur together | One stable mental model |
| Discover and validate | Bespoke calls hide incompatible arguments | Refusal before server execution |
| Survive schema drift | A redeploy changes a discovered contract | Fingerprint diff plus explicit acceptance |
| Swap providers | Agent core imports one implementation | Configuration-only provider switch |

In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "learning" / "agentic-ai" / "shared" / "__init__.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from the ai-portfolio repository or a descendant directory.")


REPO_ROOT = find_repo_root()
TRACK_DIR = REPO_ROOT / "learning" / "agentic-ai"
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from dataclasses import dataclass
from typing import Any, Callable

from pydantic import BaseModel, Field, ValidationError
from shared import INVENTORY, SUPPLIER_QUOTES

print("Local protocol simulation ready; no network transport is required.")

## 1 - Failure First: N-by-M Glue Hides Drift

With two agents and three services, bespoke adapters already create six integration relationships. Schema changes surface only when a call fails at runtime.

```mermaid
flowchart TD
    A1["Intake agent"] --> I["Inventory API"]
    A1 --> P1["Price API v1"]
    A1 --> P2["Price API v2"]
    A2["Supplier agent"] --> I
    A2 --> P1
    A2 --> P2
    style A1 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A2 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P1 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P2 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Measure bespoke integration growth ----------------------------------
def integration_edges(agent_count, service_count):
    return agent_count * service_count

baseline_edges = integration_edges(2, 3)
future_edges = integration_edges(4, 8)
print(f"Bespoke edges now: {baseline_edges}; after growth: {future_edges}")
assert future_edges > baseline_edges * 5
print("Failure observed: adding agents and services multiplies glue code.")


## 2 - Build the Protocol Lifecycle

![An OrderFlow client initializes an MCP connection, discovers versioned tool schemas, rejects an incompatible call, invokes a valid inventory tool, and can switch pricing providers through configuration](../images/ch07-mcp-discovery-lifecycle.png)

### Keep the Capability Layers Separate

| Term | Stable mental model | Authority boundary |
|---|---|---|
| **Tool** | One executable typed action | Its schema and policy gate one invocation |
| **Skill** | Reusable, versioned know-how: instructions, examples, and checks that may combine pinned tools | Grants no authority; referenced tools are still checked individually |
| **Plugin** | A framework- or host-specific package/adapter that may register tools, resources, or configuration | Installation and registration do not bypass tool policy |
| **Channel plugin** | An inbound/outbound adapter for Slack, Telegram, web, or another surface | Transports messages; it is not automatically an action tool |
| **MCP server** | A protocol endpoint publishing discoverable tools, resources, and prompts | The host validates discovered contracts and each executable call |

**Plugin is not a universal protocol primitive.** Its meaning belongs to the framework or host: Semantic Kernel, for example, uses a plugin as a named collection of callable functions, while another host may use the word for a packaging or channel adapter. MCP standardizes a client-server protocol, not a universal plugin model. Compare the registry distinctions in [Tool, MCP & Skill Registry](../../agentic-ai-system-design/03-tool-mcp-and-skill-registry.md) with the framework-specific usage in [Semantic Kernel vs. LangGraph](../../agentic-ai-system-design/13-semantic-kernel-vs-langgraph.md).

### How an Agent Call Works

The model does not directly reach across the network and run arbitrary code. It proposes a capability name and arguments; the host application owns the actual call:

1. **Choose:** the model or controller proposes a discovered capability.
2. **Package:** the MCP client creates a request containing the method, arguments, and correlation ID.
3. **Validate:** the client checks protocol version, discovered schema, and local policy before sending anything.
4. **Execute:** transport delivers the request; the server validates again and invokes the registered implementation.
5. **Observe:** the server returns a result or typed error with the same correlation ID, and the controller updates workflow state.

For a remote MCP call, authorization is deliberately two-sided. The local host confirms that this agent, tenant, and workflow may dispatch the call; the remote server independently validates its credential, scope, schema, and domain policy before performing the action. A skill pin can name the capability version, but it cannot grant either side permission.

This separation keeps generation probabilistic while execution remains attributable and policy-controlled. A retry is a new call attempt; whether it may repeat the underlying operation depends on timeout and idempotency rules explored in Chapter 09.

### How OpenClaw-Style Personal Agents Work

A "Claw" usually means an OpenClaw personal agent: a self-hosted agent harness rather than a new model or protocol. One long-running **Gateway** connects chat channels, agent sessions, models, memory, and separately sourced tools.

```mermaid
flowchart TD
    CH["Slack, Telegram,<br/>web, other channels"] --> CA["Channel adapters<br/>inbound + outbound"]
    CA --> G["Gateway<br/>auth, routing, sessions"]
    G --> C["Context<br/>conversation, memory, skills"]
    C --> M{"Model decision"}
    M -->|"Reply"| CA
    M -->|"Propose tool call"| V["Schema + policy validation"]
    DT["Direct host tools"] --> V
    PT["Plugin-provided tools"] --> V
    MT["MCP-published tools"] --> V
    V --> E["Bounded execution"]
    E --> O["Observation"]
    O --> C
    style CH fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style CA fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style DT fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style PT fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style MT fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

A typical turn follows this path:

1. **Receive:** a channel plugin accepts a message from Telegram, Slack, WhatsApp, the web UI, or another configured surface.
2. **Route:** the Gateway authenticates the sender and selects an isolated agent session and workspace.
3. **Assemble:** the agent runtime builds model context from the conversation, persistent memory, instructions, and relevant skills.
4. **Reason and act:** the model replies directly or proposes a tool call. The tool may be built into the host, registered by a framework plugin, or discovered from an MCP server. Skills guide selection and composition but grant no execution authority.
5. **Validate and execute:** every proposed tool converges on schema and policy validation before a bounded implementation runs.
6. **Observe and respond:** results update session state; the Gateway returns the final response through the originating channel. Schedules, webhooks, or heartbeats can initiate the same loop without a new chat message.

**Plugin trust:** installation and configuration, credential injection, version pinning, and process/container isolation are host or platform responsibilities. A plugin can expand what the host knows how to register, but it cannot bypass per-tool schema validation, authorization, approval, rate, or sandbox policy.

The convenience comes with a large authority surface. OpenClaw tools may run on the host unless sandboxing is configured, so channel pairing, allowlists, least-privilege credentials, approval gates, and execution isolation remain system responsibilities.

The smallest MCP-shaped lifecycle is initialize, discover, validate, call, and return a correlated result. Tools are one primitive; resources and prompts remain discoverable data, not executable authority.

```mermaid
sequenceDiagram
    participant C as Client
    participant S as Local MCP Server
    C->>S: initialize
    S-->>C: server info + protocol version
    C->>S: tools/list
    S-->>C: names + JSON schemas
    C->>S: tools/call(id, name, arguments)
    S-->>C: result(id) or typed error
```

In [ ]:
# -- Implement an in-process JSON-RPC server ------------------------------
@dataclass
class ToolSpec:
    name: str
    version: str
    input_schema: type[BaseModel]
    handler: Callable[..., dict[str, Any]]

class InventoryInput(BaseModel):
    sku: str

class PriceInputV1(BaseModel):
    sku: str

class LocalMCPServer:
    def __init__(self, name, protocol_version="2025-03-26"):
        self.name = name
        self.protocol_version = protocol_version
        self.tools = {}

    def register(self, spec):
        self.tools[spec.name] = spec

    def initialize(self):
        return {"server": self.name, "protocol_version": self.protocol_version, "capabilities": ["tools"]}

    def list_tools(self):
        return [{"name": spec.name, "version": spec.version, "schema": spec.input_schema.model_json_schema()} for spec in self.tools.values()]

    def call(self, request):
        request_id = request["id"]
        try:
            spec = self.tools[request["params"]["name"]]
            arguments = spec.input_schema.model_validate(request["params"]["arguments"])
            return {"jsonrpc": "2.0", "id": request_id, "result": spec.handler(**arguments.model_dump())}
        except (KeyError, ValidationError) as error:
            return {"jsonrpc": "2.0", "id": request_id, "error": {"code": -32602, "message": str(error)}}

inventory_server = LocalMCPServer("orderflow-inventory")
inventory_server.register(ToolSpec("inventory.lookup", "1.0.0", InventoryInput, lambda sku: {"sku": sku, "available": INVENTORY[sku]["available"]}))
print(inventory_server.initialize())
print(inventory_server.list_tools())


## 3 - Client Discovery, Validation, and Cancellation Boundaries

The client validates protocol version and input schema before calling. Unknown tools and incompatible argument shapes fail before a side effect.

```mermaid
flowchart LR
    D["Discover"] --> V{ "Compatible schema?" }
    V -->|"No"| F["Fail during validation"]
    V -->|"Yes"| C["Call with correlation ID"]
    C --> R["Result or cancellation"]
    style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Build a discovering client and prove pre-execution validation ----------
import hashlib


def schema_fingerprint(tool):
    contract = {"version": tool["version"], "schema": tool["schema"]}
    encoded = json.dumps(contract, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()[:12]


def argument_issues(schema, arguments):
    properties = schema.get("properties", {})
    missing = sorted(set(schema.get("required", [])) - set(arguments))
    unexpected = sorted(set(arguments) - set(properties))
    issues = [f"missing:{name}" for name in missing]
    issues.extend(f"unexpected:{name}" for name in unexpected)
    return issues


class MCPClient:
    def __init__(self, expected_protocol="2025-03-26"):
        self.expected_protocol = expected_protocol
        self.servers = {}
        self.catalog = {}
        self.pending = {}

    def _snapshot(self, alias):
        return {
            tool["name"]: {**tool, "fingerprint": schema_fingerprint(tool)}
            for tool in self.servers[alias].list_tools()
        }

    def connect(self, alias, server):
        info = server.initialize()
        if info["protocol_version"] != self.expected_protocol:
            raise ValueError("incompatible_protocol")
        self.servers[alias] = server
        for name, tool in self._snapshot(alias).items():
            self.catalog[f"{alias}:{name}"] = tool

    def rediscover(self, alias):
        live = self._snapshot(alias)
        self.pending[alias] = live
        differences = []
        for name, new_tool in live.items():
            old_tool = self.catalog.get(f"{alias}:{name}")
            if old_tool and old_tool["fingerprint"] == new_tool["fingerprint"]:
                continue
            old_required = set(old_tool["schema"].get("required", [])) if old_tool else set()
            new_required = set(new_tool["schema"].get("required", []))
            differences.append({
                "tool": name,
                "old_version": old_tool["version"] if old_tool else None,
                "new_version": new_tool["version"],
                "old_fingerprint": old_tool["fingerprint"] if old_tool else None,
                "new_fingerprint": new_tool["fingerprint"],
                "added_required": sorted(new_required - old_required),
                "removed_required": sorted(old_required - new_required),
            })
        return differences

    def accept_schema(self, alias, name, expected_version):
        proposed = self.pending.get(alias, {}).get(name)
        if proposed is None or proposed["version"] != expected_version:
            raise ValueError("compatibility_review_required")
        self.catalog[f"{alias}:{name}"] = proposed

    def call(self, alias, name, arguments, request_id):
        accepted = self.catalog.get(f"{alias}:{name}")
        live = self._snapshot(alias).get(name)
        if accepted is None or live is None:
            return {"jsonrpc": "2.0", "id": request_id, "error": {"code": -32601, "message": "unknown_tool", "stage": "client_catalog"}}
        if live["fingerprint"] != accepted["fingerprint"]:
            return {"jsonrpc": "2.0", "id": request_id, "error": {"code": -32010, "message": "schema_drift", "stage": "client_schema_guard"}}
        issues = argument_issues(accepted["schema"], arguments)
        if issues:
            return {"jsonrpc": "2.0", "id": request_id, "error": {"code": -32602, "message": ", ".join(issues), "stage": "client_validation"}}
        request = {"jsonrpc": "2.0", "id": request_id, "method": "tools/call", "params": {"name": name, "arguments": arguments}}
        return self.servers[alias].call(request)


client = MCPClient()
client.connect("inventory", inventory_server)
ok = client.call("inventory", "inventory.lookup", {"sku": "SKU-CPU-01"}, "req-1")
bad = client.call("inventory", "inventory.lookup", {"product_code": "SKU-CPU-01"}, "req-2")
print("Valid:", ok)
print("Refused before dispatch:", bad)
assert ok["result"]["available"] == 36
assert bad["error"]["stage"] == "client_validation"
print("PASS: incompatible arguments were refused by the client before server execution.")

### Schema Drift: Refuse, Diff, Review, Accept

**Predict:** OrderFlow cached `pricing.quote` version 1.0.0, then the server redeployed version 2.0.0 with a new required `currency` field. Should the next old-shape call (A) reach the new handler and fail there, (B) silently update the cache, or (C) stop at the client schema guard before execution?

```mermaid
flowchart LR
    A["Cached v1<br/>fingerprint"] --> D["Server deploys v2<br/>currency required"]
    D --> G{"Live fingerprint<br/>matches pin?"}
    G -->|"No"| B["Block call<br/>before execution"]
    B --> R["Rediscover +<br/>measure diff"]
    R --> P{"Compatibility<br/>review"}
    P -->|"Keep v1 pin"| B
    P -->|"Accept v2 config"| E["Validate v2 args<br/>then execute"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

The client keeps the last accepted version and canonical schema fingerprint. Rediscovery may populate a pending snapshot and produce a diff, but it does not mutate the accepted catalog. Only an explicit compatibility review and configuration update can move the pin.

In [ ]:
# -- Measure schema drift and require explicit compatibility acceptance ------
class PriceInputV2(BaseModel):
    sku: str
    currency: str = Field(pattern="^[A-Z]{3}$")


executions = {"count": 0}


def quote_v1(sku):
    executions["count"] += 1
    return {"sku": sku, "unit_price": 925.0}


def quote_v2(sku, currency):
    executions["count"] += 1
    return {"sku": sku, "currency": currency, "unit_price": 925.0}


drift_server = LocalMCPServer("pricing-drift-demo")
drift_server.register(ToolSpec("pricing.quote", "1.0.0", PriceInputV1, quote_v1))
client.connect("drift", drift_server)
accepted_v1 = client.catalog["drift:pricing.quote"]
print(f"Cached contract: version={accepted_v1['version']} fingerprint={accepted_v1['fingerprint']}")

# A deploy changes the live contract without changing the client's accepted pin.
drift_server.register(ToolSpec("pricing.quote", "2.0.0", PriceInputV2, quote_v2))
blocked = client.call("drift", "pricing.quote", {"sku": "SKU-CPU-01"}, "drift-1")
assert blocked["error"]["stage"] == "client_schema_guard"
assert executions["count"] == 0
print("Blocked after deploy:", blocked["error"]["message"], "executions=0")

schema_diff = client.rediscover("drift")
print("Measured schema diff:")
print(json.dumps(schema_diff, indent=2))
assert schema_diff[0]["old_version"] == "1.0.0"
assert schema_diff[0]["new_version"] == "2.0.0"
assert schema_diff[0]["added_required"] == ["currency"]

still_blocked = client.call("drift", "pricing.quote", {"sku": "SKU-CPU-01"}, "drift-2")
assert still_blocked["error"]["stage"] == "client_schema_guard"
assert executions["count"] == 0
print("Pinned v1 remains blocked after rediscovery; rediscovery alone accepted nothing.")

client.accept_schema("drift", "pricing.quote", expected_version="2.0.0")
old_shape = client.call("drift", "pricing.quote", {"sku": "SKU-CPU-01"}, "drift-3")
assert old_shape["error"]["stage"] == "client_validation"
assert executions["count"] == 0

accepted = client.call("drift", "pricing.quote", {"sku": "SKU-CPU-01", "currency": "USD"}, "drift-4")
assert accepted["result"]["currency"] == "USD"
assert executions["count"] == 1
print("PASS: review accepted v2; only v2-shaped arguments reached execution.")

**Reflection:** The first post-deploy call left the execution count at zero because the live fingerprint no longer matched the accepted pin. Rediscovery measured the contract change (`1.0.0` to `2.0.0`, new fingerprint, required `currency`) but deliberately left the old pin active. After explicit acceptance, the old argument shape still failed client validation; only the reviewed v2 shape reached the handler. Discovery tells you what exists. Review decides what may replace an accepted contract.

## 4 - Add a Provider Through Configuration Alone

Agent-core code asks the client catalog for a configured pricing alias. The new provider implements the same versioned contract; the agent does not import provider-specific functions.

```mermaid
flowchart LR
    A["OrderFlow agent"] --> C["MCP client catalog"]
    C --> P1["primary pricing server"]
    C --> P2["backup pricing server"]
    CFG["configuration"] --> C
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P1 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P2 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style CFG fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Register two compatible providers and switch only configuration ------
def make_pricing_server(name, adjustment):
    server = LocalMCPServer(name)
    def price_lookup(sku):
        trusted = [quote for quote in SUPPLIER_QUOTES[sku] if quote["trusted"] and quote["age_hours"] <= 48]
        best = min(trusted, key=lambda item: item["unit_price"])
        return {"sku": sku, "supplier": best["supplier"], "unit_price": round(best["unit_price"] + adjustment, 2)}
    server.register(ToolSpec("pricing.quote", "1.0.0", PriceInputV1, price_lookup))
    return server

client.connect("primary", make_pricing_server("pricing-primary", 0.0))
client.connect("backup", make_pricing_server("pricing-backup", 3.0))

def agent_price_lookup(client, config, sku):
    alias = config["pricing_provider"]
    return client.call(alias, "pricing.quote", {"sku": sku}, f"price-{sku}")["result"]

primary = agent_price_lookup(client, {"pricing_provider": "primary"}, "SKU-CPU-01")
backup = agent_price_lookup(client, {"pricing_provider": "backup"}, "SKU-CPU-01")
print("Primary:", primary)
print("Backup:", backup)
assert primary["unit_price"] != backup["unit_price"]
print("PASS: provider changed through configuration without changing agent_price_lookup.")


## Roadmap Checkpoint

```mermaid
flowchart LR
    A["Capability layers<br/>separated"] --> B["Schema drift<br/>blocked + diffed"]
    B --> C["Provider switched<br/>by configuration"]
    C --> D["Next: multi-agent<br/>coordination"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Before | After |
|---|---:|---:|
| Capability vocabulary | Tool, skill, plugin, channel plugin, and MCP server blurred | Distinct roles and authority boundaries |
| Integration edges | N agents x M services | One client lifecycle per server |
| Capability discovery | Hard-coded imports | Versioned catalog with schema fingerprint |
| Invalid arguments | Server-side surprise | Client refusal before dispatch |
| Schema drift | Redeploy could silently change a contract | Zero executions until diff, review, and explicit acceptance |
| New pricing provider | Agent edit | Configuration-only switch |

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Lifecycle, tool discovery, JSON-RPC correlation, client schema validation, schema fingerprint/version drift guard, measured rediscovery diff, explicit acceptance, provider switch |
| Explained and illustrated | Tool/skill/plugin/channel-plugin/MCP-server boundaries, agent-call ownership, OpenClaw Gateway anatomy, plugin trust responsibilities, resources, prompts, cancellation, transports, authorization |
| Named with a reason | Remote HTTP transport and official SDK, deferred to keep the core offline and deterministic |

### Key Takeaways

- A tool is one typed executable action; a skill is versioned know-how and grants no authority.
- Plugin is a framework- or host-specific packaging term, not a universal protocol primitive.
- Channel plugins transport messages; direct, plugin-provided, and MCP-published tools still converge on the same schema and policy gate.
- An MCP server publishes discoverable tools, resources, and prompts; discovery is not acceptance.
- Cache versioned schema fingerprints, refuse drift before execution, diff rediscovery results, and update pins only after explicit compatibility review.
- Installation, credentials, pinning, and isolation belong to the host or platform; plugins do not bypass tool policy.
- Correlation IDs make concurrent calls attributable, and configuration can switch compatible providers without agent-core edits.
- Use a normal function call when process or ownership boundaries do not exist.